In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
from torchvision import models, transforms
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 72
IMG_SIZE = 224

from dataloader.data_module import DataModule
from dataloader.classification_dataset import CheXpertImageDataset
from dataloader.transforms import DataTransforms, Moco2Transform
datamodule = DataModule(CheXpertImageDataset, None,
                        Moco2Transform, data_pct=1.0,
                        batch_size=BATCH_SIZE, num_workers=16)
dataloader = datamodule.val_dataloader()

from models.main_net import main_net
BASE_DIR = os.getcwd()
baseline_model_path = BASE_DIR+"path/to/baseline/checkpoint"
CoMA_model_path = BASE_DIR+"path/to/CoMA/checkpoint"
baseline_model = main_net.load_from_checkpoint(baseline_model_path, strict=False).to(device)
CoMA_model = main_net.load_from_checkpoint(CoMA_model_path, strict=False).to(device)

In [ ]:
from constants import *
def get_tsne_feature(features_baseline, features_coma, perplexity=30):
    # 执行t-SNE降维
    print("Performing t-SNE for baseline features...")
    tsne_baseline = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
    features_2d_baseline = tsne_baseline.fit_transform(features_baseline)
    
    print("Performing t-SNE for CoMA features...")
    tsne_coma = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
    features_2d_coma = tsne_coma.fit_transform(features_coma)
    
    return features_2d_baseline, features_2d_coma
    
def plot_tsne_comparison(features_2d_baseline, features_2d_coma, labels, title_suffix=""):
    """对比显示两个模型的t-SNE可视化结果"""
    
    # 获取唯一的标签和颜色
    unique_labels = sorted(np.unique(labels))
    colors = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
    color_map = {label: colors[i] for i, label in enumerate(unique_labels)}
    
    # 创建对比图
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # 绘制baseline结果
    for label in unique_labels:
        mask = np.array(labels) == label
        strlabel = CHEXPERT_COMPETITION_TASKS[label]
        ax1.scatter(features_2d_baseline[mask, 0], features_2d_baseline[mask, 1],
                   c=[color_map[label]], label=strlabel, alpha=0.7, s=30)
    
    ax1.set_title(f'Baseline', fontsize=30)
    ax1.set_xlabel('t-SNE Component 1', fontsize=20)
    ax1.set_ylabel('t-SNE Component 2', fontsize=20)
    # ax1.legend(bbox_to_anchor=(0.85, 1), loc='upper left')
    ax1.legend(loc='upper right', fontsize=15)
    ax1.grid(True, alpha=0.3)
    
    # 绘制CoMA结果
    for label in unique_labels:
        mask = np.array(labels) == label
        strlabel = CHEXPERT_COMPETITION_TASKS[label]
        ax2.scatter(features_2d_coma[mask, 0], features_2d_coma[mask, 1],
                   c=[color_map[label]], label=strlabel, alpha=0.7, s=30)
    
    ax2.set_title(f'CoMA', fontsize=30)
    ax2.set_xlabel('t-SNE Component 1', fontsize=20)
    ax2.set_ylabel('t-SNE Component 2', fontsize=20)
    # ax2.legend(bbox_to_anchor=(0.85, 1), loc='upper left')
    ax2.legend(loc='upper right', fontsize=15)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return features_2d_baseline, features_2d_coma

In [ ]:
from tqdm.contrib import tenumerate
def extract_features_batch(model, dataloader):
    features_list = []
    valid_labels = []
    for i,batch in tenumerate(dataloader):
        x, y = batch
        # For multi-class
        with torch.no_grad():
            feats, _, _ = model.img_encoder_q(x.to(device), model.propompt_vector)
        feats = feats.view(feats.size(0), -1)
        features_list.append(feats.cpu().numpy())
        valid_labels.append(np.array(y))
        # if i>=20:
        #     break
    features_list= np.vstack(features_list)
    valid_labels= np.vstack(valid_labels)
    return features_list, valid_labels

In [ ]:
baseline_features, baseline_labels = extract_features_batch(baseline_model, dataloader)
CoMA_features, CoMA_labels = extract_features_batch(CoMA_model, dataloader)
if not np.array_equal(baseline_labels, CoMA_labels):
    print("labels not match")        
    print(baseline_labels)
    print(CoMA_labels)

In [ ]:
def sample_filter(baseline_features, CoMA_features, labels, n_sample=None):
    mask = np.array([np.sum(s)==1 for s in labels])
    baseline_features = baseline_features[mask]
    CoMA_features = CoMA_features[mask]
    labels = labels[mask]
    n_labels = np.argmax(labels, axis=1)
    
    if n_sample is not None:
        print('now do sampling')
        _,c = labels.shape
        mask = []
        n_class = [0 for _ in range(c)]
        for s in n_labels:
            if n_class[s]<n_sample:
                mask.append(True)
                n_class[s]+=1
            else:
                mask.append(False)
        mask = np.array(mask)
        baseline_features = baseline_features[mask]
        CoMA_features = CoMA_features[mask]
        n_labels = n_labels[mask]
    return baseline_features, CoMA_features, n_labels

n_baseline_features, n_CoMA_features, n_labels = sample_filter(baseline_features, CoMA_features, baseline_labels, n_sample=1000)

In [ ]:
features_2d_baseline, features_2d_coma = get_tsne_feature(n_baseline_features, n_CoMA_features, perplexity=20)
features_2d_baseline, features_2d_coma = plot_tsne_comparison(features_2d_baseline, features_2d_coma, n_labels, title_suffix="")